In [12]:
pip install xlrd openpyxl

  Obtaining dependency information for xlrd from https://files.pythonhosted.org/packages/1a/62/c8d562e7766786ba6587d09c5a8ba9f718ed3fa8af7f4553e8f91c36f302/xlrd-2.0.2-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os

# Documents not Desktop
base = os.path.expanduser("~/Documents/NHS_Project")

for folder in ["RTT_full", "RTT_incomplete", "AE"]:
    files = os.listdir(os.path.join(base, folder))
    print(f"\n{folder}:")
    for f in files:
        print(f"  {f}")


RTT_full:
  20251031-RTT-October-2025-full-extract.csv
  20251130-RTT-November-2025-full-extract.csv
  20260228-RTT-February-2026-full-extract.csv
  20260331-RTT-March-2026-full-extract.csv
  20251231-RTT-December-2025-full-extract.csv
  20260131-RTT-January-2026-full-extract.csv

RTT_incomplete:
  Incomplete-Provider-Dec25-XLSX-9M-6jPlxd.xlsx
  Incomplete-Provider-Jan26-XLSX-9M-WL5BiP.xlsx
  Incomplete-Provider-Mar26-XLSX-9M-Dc1i9U.xlsx
  Incomplete-Provider-Oct25-XLSX-9M-SrRW6y.xlsx
  Incomplete-Provider-Nov25-XLSX-9M-1Xmjkk.xlsx
  Incomplete-Provider-Feb26-XLSX-9M-9j03fJT.xlsx

AE:
  Quarter-4-2025-26-AE-by-provider-7QC8I.xls
  Monthly-AE-March-2026-revised-flkg42.csv
  Quarter-3-2025-26-AE-by-provider-revised-qe43dg.xls


In [4]:
sample = pd.read_csv(os.path.join(base, "RTT_full", "20251031-RTT-October-2025-full-extract.csv"))
print(sample.shape)
print(sample.columns.tolist())
sample.head(3)

(185742, 121)
['Period', 'Provider Parent Org Code', 'Provider Parent Name', 'Provider Org Code', 'Provider Org Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name', 'Commissioner Org Code', 'Commissioner Org Name', 'RTT Part Type', 'RTT Part Description', 'Treatment Function Code', 'Treatment Function Name', 'Gt 00 To 01 Weeks SUM 1', 'Gt 01 To 02 Weeks SUM 1', 'Gt 02 To 03 Weeks SUM 1', 'Gt 03 To 04 Weeks SUM 1', 'Gt 04 To 05 Weeks SUM 1', 'Gt 05 To 06 Weeks SUM 1', 'Gt 06 To 07 Weeks SUM 1', 'Gt 07 To 08 Weeks SUM 1', 'Gt 08 To 09 Weeks SUM 1', 'Gt 09 To 10 Weeks SUM 1', 'Gt 10 To 11 Weeks SUM 1', 'Gt 11 To 12 Weeks SUM 1', 'Gt 12 To 13 Weeks SUM 1', 'Gt 13 To 14 Weeks SUM 1', 'Gt 14 To 15 Weeks SUM 1', 'Gt 15 To 16 Weeks SUM 1', 'Gt 16 To 17 Weeks SUM 1', 'Gt 17 To 18 Weeks SUM 1', 'Gt 18 To 19 Weeks SUM 1', 'Gt 19 To 20 Weeks SUM 1', 'Gt 20 To 21 Weeks SUM 1', 'Gt 21 To 22 Weeks SUM 1', 'Gt 22 To 23 Weeks SUM 1', 'Gt 23 To 24 Weeks SUM 1', 'Gt 24 To 25 Weeks SUM 1', '

,Period,Provider Parent Org Code,Provider Parent Name,Provider Org Code,Provider Org Name,Commissioner Parent Org Code,Commissioner Parent Name,Commissioner Org Code,Commissioner Org Name,RTT Part Type,...,Gt 98 To 99 Weeks SUM 1,Gt 99 To 100 Weeks SUM 1,Gt 100 To 101 Weeks SUM 1,Gt 101 To 102 Weeks SUM 1,Gt 102 To 103 Weeks SUM 1,Gt 103 To 104 Weeks SUM 1,Gt 104 Weeks SUM 1,Total,Patients with unknown clock start date,Total All
0,RTT-October-2025,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1
1,RTT-October-2025,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1
2,RTT-October-2025,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,Y62,NaN,Part_1B,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1


In [8]:
import glob

rtt_files = glob.glob(os.path.join(base, "RTT_full", "*.csv"))
rtt_all = pd.concat([pd.read_csv(f) for f in rtt_files], ignore_index=True)

keep_cols = [
    "Period",
    "Provider Org Code",
    "Provider Org Name",
    "Commissioner Parent Name",
    "RTT Part Description",
    "Treatment Function Name",
    "Total",
    "Total All"
]

within_18_cols = [c for c in rtt_all.columns if any(
    c.startswith(f"Gt {str(i).zfill(2)}") for i in range(18)
)]

rtt = rtt_all[keep_cols].copy()
rtt["Within_18_Weeks"] = rtt_all[within_18_cols].sum(axis=1)
rtt["Pct_Within_18_Weeks"] = (rtt["Within_18_Weeks"] / rtt["Total All"] * 100).round(2)
rtt["Period"] = pd.to_datetime(rtt["Period"], format="RTT-%B-%Y")

print("Shape:", rtt.shape)
print("\nSample:")
rtt.head(3)

Shape: (1093888, 10)

Sample:


,Period,Provider Org Code,Provider Org Name,Commissioner Parent Name,RTT Part Description,Treatment Function Name,Total,Total All,Within_18_Weeks,Pct_Within_18_Weeks
0,2025-10-01,A4M8P,BUCKSHAW HOSPITAL,NaN,Incomplete Pathways,Urology Service,NaN,1,1.0,100.0
1,2025-10-01,A4M8P,BUCKSHAW HOSPITAL,NaN,Incomplete Pathways,Total,NaN,1,1.0,100.0
2,2025-10-01,A4M8P,BUCKSHAW HOSPITAL,NaN,Completed Pathways For Non-Admitted Patients,General Surgery Service,1.0,1,1.0,100.0


In [9]:
# Keep only Incomplete Pathways (patients still waiting - most important for dashboard)
rtt_clean = rtt[rtt["RTT Part Description"] == "Incomplete Pathways"].copy()

# Remove rows where Treatment Function is "Total" (summary rows, we don't need them)
rtt_clean = rtt_clean[rtt_clean["Treatment Function Name"] != "Total"]

# Drop rows with no patients
rtt_clean = rtt_clean[rtt_clean["Total All"] > 0]

# Reset index
rtt_clean = rtt_clean.reset_index(drop=True)

print("Final shape:", rtt_clean.shape)
print("\nMonths included:")
print(rtt_clean["Period"].dt.strftime("%B %Y").unique())
print("\nTop 10 specialties by volume:")
print(rtt_clean.groupby("Treatment Function Name")["Total All"].sum().sort_values(ascending=False).head(10))

Final shape: (310306, 10)

Months included:
['October 2025' 'November 2025' 'February 2026' 'March 2026'
 'December 2025' 'January 2026']

Top 10 specialties by volume:
Treatment Function Name
Trauma and Orthopaedic Service    5083988
Other - Medical Services          3732675
Ophthalmology Service             3609117
Ear Nose and Throat Service       3553118
Gynaecology Service               3401308
Other - Surgical Services         2872005
Dermatology Service               2405099
Urology Service                   2321253
General Surgery Service           2308601
Cardiology Service                2305531
Name: Total All, dtype: int64


In [10]:
# Save clean RTT data for Tableau
output_path = os.path.join(base, "rtt_clean.csv")
rtt_clean.to_csv(output_path, index=False)
print("RTT data saved!")
print(f"Location: {output_path}")
print(f"Rows: {len(rtt_clean):,}")
print(f"Columns: {list(rtt_clean.columns)}") 

RTT data saved!
Location: /Users/marselahalili/Documents/NHS_Project/rtt_clean.csv
Rows: 310,306
Columns: ['Period', 'Provider Org Code', 'Provider Org Name', 'Commissioner Parent Name', 'RTT Part Description', 'Treatment Function Name', 'Total', 'Total All', 'Within_18_Weeks', 'Pct_Within_18_Weeks']


In [13]:
import glob

# Load A&E quarter files
ae_files = glob.glob(os.path.join(base, "AE", "*.xls")) + \
           glob.glob(os.path.join(base, "AE", "*.csv"))

print("A&E files found:")
for f in ae_files:
    print(f" {os.path.basename(f)}")

# Peek at the first file
ae_sample = pd.read_excel(ae_files[0], header=None)
print("\nFirst 10 rows to understand structure:")
ae_sample.head(10)
 

A&E files found:
 Quarter-4-2025-26-AE-by-provider-7QC8I.xls
 Quarter-3-2025-26-AE-by-provider-revised-qe43dg.xls
 Monthly-AE-March-2026-revised-flkg42.csv

First 10 rows to understand structure:


,0,1,2,3,4,5,6,7,8,9
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Org code,Region,Name,STP,NaN,NaN,STP,Code,NaN,NaN
3,RF4,London Commissioning Region,"Barking, Havering and Redbridge University Hos...",NHS North East London Integrated Care Board,NaN,NaN,"Bath and North East Somerset, Swindon and Wilt...",QOX,"NHS BATH AND NORTH EAST SOMERSET, SWINDON AND ...","NHS Bath And North East Somerset, Swindon And ..."
4,R1H,London Commissioning Region,Barts Health NHS Trust,NHS North East London Integrated Care Board,NaN,NaN,"Bedfordshire, Luton and Milton Keynes STP",QHG,"NHS BEDFORDSHIRE, LUTON AND MILTON KEYNES INTE...","NHS Bedfordshire, Luton And Milton Keynes Inte..."
5,Y03082,London Commissioning Region,Harold Wood Walk In Centre,NHS North East London Integrated Care Board,NaN,NaN,Birmingham and Solihull STP,QHL,NHS BIRMINGHAM AND SOLIHULL INTEGRATED CARE BOARD,NHS Birmingham And Solihull Integrated Care Board
6,RQX,London Commissioning Region,Homerton Healthcare NHS Foundation Trust,NHS North East London Integrated Care Board,NaN,NaN,"Bristol, North Somerset and South Gloucestersh...",QUY,"NHS BRISTOL, NORTH SOMERSET AND SOUTH GLOUCEST...","NHS Bristol, North Somerset And South Gloucest..."
7,RAT,London Commissioning Region,North East London NHS Foundation Trust,NHS North East London Integrated Care Board,NaN,NaN,"Buckinghamshire, Oxfordshire and Berkshire Wes...",QU9,"NHS BUCKINGHAMSHIRE, OXFORDSHIRE AND BERKSHIRE...","NHS Buckinghamshire, Oxfordshire And Berkshire..."
8,Y02973,London Commissioning Region,Orchard Village Walk-In-Centre,NHS North East London Integrated Care Board,NaN,NaN,Cambridgeshire and Peterborough STP,QUE,NHS CAMBRIDGESHIRE AND PETERBOROUGH INTEGRATED...,NHS Cambridgeshire And Peterborough Integrated...
9,Y03047,London Commissioning Region,St Andrews Walk-In Centre,NHS North East London Integrated Care Board,NaN,NaN,Cheshire and Merseyside STP,QYG,NHS CHESHIRE AND MERSEYSIDE INTEGRATED CARE BOARD,NHS Cheshire And Merseyside Integrated Care Board


In [14]:
# Look further down the file to find the attendance columns
ae_sample2 = pd.read_excel(ae_files[0], header=None)
print("Total rows:", len(ae_sample2))
print("Total cols:", len(ae_sample2.columns))
print("\nRow 2 (headers):")
print(ae_sample2.iloc[2].tolist())

Total rows: 275
Total cols: 10

Row 2 (headers):
['Org code', 'Region ', 'Name', 'STP', nan, nan, 'STP', 'Code', nan, nan]


In [15]:
# Check all sheet names in the file
import xlrd
wb = xlrd.open_workbook(ae_files[0])
print("Sheets:", wb.sheet_names())

Sheets: ['System Mapping', 'System Level Data', 'Provider Level Data', 'Non-Booked Data', 'Booked Appointments Data', 'Acute Trust Footprint Data', 'Acute Trust Mapping']


In [16]:
# Read the Provider Level Data sheet
ae_provider = pd.read_excel(ae_files[0], sheet_name="Provider Level Data", header=None)
print("Shape:", ae_provider.shape)
print("\nFirst 5 rows:")
ae_provider.head(5)

Shape: (230, 29)

First 5 rows:


,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,NaN,NaN,NaN,NaN,4,5,6,NaN,NaN,NaN,...,NaN,18,19,20,NaN,21,NaN,16,17,NaN
1,NaN,Title:,A&E Attendances & Emergency Admission monthly ...,NaN,7,8,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Summary:,The total of booked and non-booked A&E attenda...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Print every row until we find the real headers
for i in range(20):
    print(f"Row {i}: {ae_provider.iloc[i].tolist()}")

Row 0: [nan, nan, nan, nan, 4, 5, 6, nan, nan, nan, nan, nan, 10, 11, 12, nan, nan, nan, nan, nan, 18, 19, 20, nan, 21, nan, 16, 17, nan]
Row 1: [nan, 'Title:', 'A&E Attendances & Emergency Admission monthly statistics, NHS and independent sector organisations in England', nan, 7, 8, 9, nan, nan, nan, nan, nan, 13, 14, 15, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Row 2: [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Row 3: [nan, 'Summary:', 'The total of booked and non-booked A&E attendances, as well as emergency admissions and headline 4-hour performance.\nPlease see the Supplementary ECDS Analysis file for further breakdowns of A&E attendance data, this can be found published below this file on the NHS England Website, this provides attendance data split by age groups, gender, ethnicity, chief complaint and provides a data completeness and quality fact sheet

In [18]:
# Read A&E data properly from both quarter files
ae_dfs = []

for f in ae_files:
    if f.endswith('.xls'):
        df = pd.read_excel(f, sheet_name="Provider Level Data", header=15)
        # Add filename so we know which quarter it came from
        df["source_file"] = os.path.basename(f)
        ae_dfs.append(df)

ae = pd.concat(ae_dfs, ignore_index=True)

# Keep only the columns we need
ae = ae[[
    ae.columns[1],   # Code
    ae.columns[2],   # Region
    ae.columns[3],   # Name
    ae.columns[7],   # Total attendances
    ae.columns[11],  # Total < 4 hours
    ae.columns[15],  # Total > 4 hours
    ae.columns[16],  # % within 4 hours
    ae.columns[25],  # Total Emergency Admissions
    "source_file"
]].copy()

# Rename columns cleanly
ae.columns = [
    "Org_Code", "Region", "Provider_Name",
    "Total_Attendances", "Within_4hrs", "Over_4hrs",
    "Pct_Within_4hrs", "Total_Emergency_Admissions",
    "Source_File"
]

# Remove summary/blank rows
ae = ae[ae["Org_Code"].notna()]
ae = ae[ae["Org_Code"] != "-"]

print("Shape:", ae.shape)
print("\nSample:")
ae.head(5)

Shape: (396, 9)

Sample:


,Org_Code,Region,Provider_Name,Total_Attendances,Within_4hrs,Over_4hrs,Pct_Within_4hrs,Total_Emergency_Admissions,Source_File
2,RC9,NHS England East Of England,Bedfordshire Hospitals NHS Foundation Trust,78705.0,61161.0,17544.0,0.777092,23090.0,Quarter-4-2025-26-AE-by-provider-7QC8I.xls
3,RGT,NHS England East Of England,Cambridge University Hospitals NHS Foundation ...,50512.0,35127.0,15385.0,0.695419,9950.0,Quarter-4-2025-26-AE-by-provider-7QC8I.xls
4,RWH,NHS England East Of England,East And North Hertfordshire NHS Trust,52403.0,37097.0,15306.0,0.707917,7602.0,Quarter-4-2025-26-AE-by-provider-7QC8I.xls
5,RDE,NHS England East Of England,East Suffolk And North Essex NHS Foundation Trust,71599.0,54180.0,17419.0,0.756714,15525.0,Quarter-4-2025-26-AE-by-provider-7QC8I.xls
6,RY4,NHS England East Of England,Hertfordshire Community NHS Trust,3284.0,3229.0,55.0,0.983252,0.0,Quarter-4-2025-26-AE-by-provider-7QC8I.xls


In [19]:
# Convert % to readable percentage
ae["Pct_Within_4hrs"] = (ae["Pct_Within_4hrs"] * 100).round(2)

# Save A&E clean data
ae_output = os.path.join(base, "ae_clean.csv")
ae.to_csv(ae_output, index=False)
print("A&E data saved!")
print(f"Location: {ae_output}")
print(f"Rows: {len(ae):,}")

# Final summary
print("\n--- YOUR DASHBOARD DATA IS READY ---")
print(f"RTT data: {len(rtt_clean):,} rows across 6 months")
print(f"A&E data: {len(ae):,} rows across 2 quarters")
print("\nFiles saved to:")
print(f"  {os.path.join(base, 'rtt_clean.csv')}")
print(f"  {os.path.join(base, 'ae_clean.csv')}")

TypeError: can't multiply sequence by non-int of type 'float'

In [20]:
# Force numeric, any text like "-" becomes NaN
ae["Pct_Within_4hrs"] = pd.to_numeric(ae["Pct_Within_4hrs"], errors="coerce")
ae["Pct_Within_4hrs"] = (ae["Pct_Within_4hrs"] * 100).round(2)

# Save A&E clean data
ae_output = os.path.join(base, "ae_clean.csv")
ae.to_csv(ae_output, index=False)
print("A&E data saved!")
print(f"Rows: {len(ae):,}")

print("\n--- YOUR DASHBOARD DATA IS READY ---")
print(f"RTT data: {len(rtt_clean):,} rows across 6 months")
print(f"A&E data: {len(ae):,} rows across 2 quarters")
print("\nFiles saved to:")
print(f"  {os.path.join(base, 'rtt_clean.csv')}")
print(f"  {os.path.join(base, 'ae_clean.csv')}")

A&E data saved!
Rows: 396

--- YOUR DASHBOARD DATA IS READY ---
RTT data: 310,306 rows across 6 months
A&E data: 396 rows across 2 quarters

Files saved to:
  /Users/marselahalili/Documents/NHS_Project/rtt_clean.csv
  /Users/marselahalili/Documents/NHS_Project/ae_clean.csv
